# 05 - Gold Layer Checks
Validates the star schema: keys, SCD2 integrity, point-in-time joins, and a sample business query.

In [0]:
%sql
SELECT
  -- every fact row found its customer and product
  (SELECT COUNT(*) FROM retail_lakehouse.gold.fact_orders WHERE customer_sk IS NULL) AS facts_missing_customer,
  (SELECT COUNT(*) FROM retail_lakehouse.gold.fact_orders WHERE product_sk  IS NULL) AS facts_missing_product,
  -- SCD2: every customer has exactly ONE current version
  (SELECT COUNT(*) FROM (
      SELECT customer_id FROM retail_lakehouse.gold.dim_customer
      GROUP BY customer_id
      HAVING SUM(CASE WHEN is_current THEN 1 ELSE 0 END) <> 1
  )) AS customers_without_one_current_row,
  -- no duplicate fact rows
  (SELECT COUNT(*) FROM (
      SELECT order_id, order_item_id FROM retail_lakehouse.gold.fact_orders
      GROUP BY order_id, order_item_id HAVING COUNT(*) > 1
  )) AS duplicate_fact_rows;

In [0]:
%sql
SELECT customer_sk, customer_id, customer_city, customer_state, customer_zip_code_prefix,
       effective_from, effective_to, is_current
FROM retail_lakehouse.gold.dim_customer
WHERE customer_id IN (
  SELECT customer_id FROM retail_lakehouse.gold.dim_customer
  GROUP BY customer_id HAVING COUNT(*) > 1
)
ORDER BY customer_id, effective_from;

In [0]:
%sql
SELECT f.order_id, f.order_purchase_timestamp, f.item_total,
       d.customer_sk, d.customer_city, d.customer_state, d.is_current
FROM retail_lakehouse.gold.fact_orders f
JOIN retail_lakehouse.gold.dim_customer d ON f.customer_sk = d.customer_sk
WHERE d.customer_id IN (
  SELECT customer_id FROM retail_lakehouse.gold.dim_customer
  GROUP BY customer_id HAVING COUNT(*) > 1
);

In [0]:
%sql
SELECT
  d.year,
  d.month,
  p.product_category_name,
  COUNT(DISTINCT f.order_id)  AS orders,
  ROUND(SUM(f.item_total), 2) AS revenue,
  ROUND(AVG(f.delivery_days), 1) AS avg_delivery_days,
  ROUND(AVG(CAST(f.is_late_delivery AS INT)) * 100, 1) AS late_delivery_pct
FROM retail_lakehouse.gold.fact_orders f
JOIN retail_lakehouse.gold.dim_date d    ON f.purchase_date_key = d.date_key
JOIN retail_lakehouse.gold.dim_product p ON f.product_sk = p.product_sk
WHERE d.year = 2018
GROUP BY d.year, d.month, p.product_category_name
ORDER BY revenue DESC
LIMIT 15;